# Geographic Hotspot Identification for Migration

## Description
This project will explore how visualization and spatial analytics can reveal geographic “hotspots” of migration risk using open data. The core idea is to combine multiple socioeconomic and environmental indicators — such as poverty, unemployment, education levels, remittances, and climate risk — into a composite migration-risk score for each region. The focous will be Latin America and the Caribbean.

In [1]:
# imports
import altair as alt
alt.data_transformers.disable_max_rows()
import polars as pl
from pathlib import Path
from vega_datasets import data
import geopandas as gpd
import os, sys
sys.path.insert(0, os.path.abspath("..")) 



In [2]:
migrants_stock = pl.read_csv(Path("../Data/migrants_stock_undesa.csv"))
migrants_stock

index,destination,coverage,type,destination_code,origin,origin_code,gender,1990,1995,2000,2005,2010,2015,2020,2024
i64,str,str,str,i64,str,i64,str,str,str,str,str,str,str,str,str
1,"""World""",null,null,900,"""World""",900,"""male""","""77 772 082""","""82 543 298""","""88 290 942""","""98 450 665""","""113 256 786""","""130 187 624""","""143 223 497""","""158 009 795"""
2,"""World""",null,null,900,"""Sub-Saharan Africa""",1834,"""male""","""7 458 958""","""8 020 179""","""7 737 985""","""8 678 854""","""9 739 644""","""12 066 851""","""14 375 255""","""16 046 945"""
3,"""World""",null,null,900,"""Northern Africa and Western As…",1833,"""male""","""8 337 422""","""9 668 614""","""10 647 817""","""12 168 680""","""14 628 780""","""18 783 440""","""21 212 741""","""22 913 639"""
4,"""World""",null,null,900,"""Central and Southern Asia""",1831,"""male""","""16 862 386""","""15 721 261""","""17 029 411""","""18 867 466""","""23 546 473""","""28 740 559""","""30 263 324""","""33 260 124"""
5,"""World""",null,null,900,"""Eastern and South-Eastern Asia""",1832,"""male""","""7 069 837""","""8 373 109""","""9 761 566""","""11 238 493""","""13 840 573""","""15 768 936""","""17 926 170""","""19 677 034"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
28026,"""Wallis and Futuna Islands*""","""44""","""B R""",876,"""New Caledonia*""",540,"""female""","""410""","""480""","""574""","""598""","""545""","""513""","""512""","""510"""
28027,"""Wallis and Futuna Islands*""","""44""","""B R""",876,"""Vanuatu""",548,"""female""","""73""","""80""","""96""","""97""","""68""","""34""","""28""","""28"""
28028,"""Wallis and Futuna Islands*""","""44""","""B R""",876,"""Polynesia*""",957,"""female""","""8""","""11""","""13""","""16""","""9""","""7""","""5""","""5"""


In [3]:
country_codes = pl.read_csv(Path("../Data/country_codes.csv"),ignore_errors=True)
country_codes


name,alpha-2,alpha-3,country-code,M49,iso_3166-2,region,sub-region,intermediate-region,region-code,sub-region-code,intermediate-region-code
str,str,str,i64,i64,str,str,str,str,i64,i64,i64
"""Afghanistan""","""AF""","""AFG""",4,4,"""ISO 3166-2:AF""","""Asia""","""Southern Asia""",null,142,34,null
"""Åland Islands""","""AX""","""ALA""",248,248,"""ISO 3166-2:AX""","""Europe""","""Northern Europe""",null,150,154,null
"""Albania""","""AL""","""ALB""",8,8,"""ISO 3166-2:AL""","""Europe""","""Southern Europe""",null,150,39,null
"""Algeria""","""DZ""","""DZA""",12,12,"""ISO 3166-2:DZ""","""Africa""","""Northern Africa""",null,2,15,null
"""American Samoa""","""AS""","""ASM""",16,16,"""ISO 3166-2:AS""","""Oceania""","""Polynesia""",null,9,61,null
…,…,…,…,…,…,…,…,…,…,…,…
"""Wallis and Futuna""","""WF""","""WLF""",876,876,"""ISO 3166-2:WF""","""Oceania""","""Polynesia""",null,9,61,null
"""Western Sahara""","""EH""","""ESH""",732,732,"""ISO 3166-2:EH""","""Africa""","""Northern Africa""",null,2,15,null
"""Yemen""","""YE""","""YEM""",887,887,"""ISO 3166-2:YE""","""Asia""","""Western Asia""",null,142,145,null


In [4]:
# Our data base is in a wide format, we need to convert it to a long format
migrants = migrants_stock.melt(
    id_vars=[
        "index", "destination", "coverage", "type", 
        "destination_code", "origin", "origin_code", "gender"
    ],
    value_vars=["1990", "1995", "2000", "2005", "2010", "2015", "2020", "2024"],
    variable_name="year",
    value_name="migrants"    
)

migrants = migrants.with_columns([
        # year → integer
        pl.col("year").cast(pl.Int32),

        # clean and cast migrants
        (
            pl.col("migrants")
            .cast(pl.Utf8)                               # ensure string ops ok
            # turn common placeholders into nulls
            .replace({"..": None, "—": None, "-": None, "": None, " ":""})
            # remove commas/spaces/other non-digits
            .str.replace_all(r"[^\d.]", "")
            # empty string after cleaning → null
            .map_elements(lambda s: None if s == "" else s, return_dtype=pl.Utf8)
            # finally cast to number; strict=False turns bad leftovers into null
            .cast(pl.Int64, strict=False)
        ).alias("migrants"),

        # clean country orijin and destination names
        pl.col("origin").str.replace_all(r"\*", ""),
        pl.col("destination").str.replace_all(r"\*", ""),

        pl.col("origin_code").alias("M49")

    ])

migrants = migrants.with_columns((pl.col("migrants") / 1_000_000).alias("migrants_millions"))




/var/folders/2l/np27k3x96zlgg9n3dvyy9hxh0000gn/T/ipykernel_40795/1424073704.py:2: DeprecationWarning: `DataFrame.melt` is deprecated; use `DataFrame.unpivot` instead, with `index` instead of `id_vars` and `on` instead of `value_vars`
  migrants = migrants_stock.melt(


In [5]:
#we really need to be more carefull about the data 
# we have destinations incluidng agregates like "World" or "High income countries"
# we will filter only the destinations that are countries
migrants

index,destination,coverage,type,destination_code,origin,origin_code,gender,year,migrants,M49,migrants_millions
i64,str,str,str,i64,str,i64,str,i32,i64,i64,f64
1,"""World""",null,null,900,"""World""",900,"""male""",1990,77772082,900,77.772082
2,"""World""",null,null,900,"""Sub-Saharan Africa""",1834,"""male""",1990,7458958,1834,7.458958
3,"""World""",null,null,900,"""Northern Africa and Western As…",1833,"""male""",1990,8337422,1833,8.337422
4,"""World""",null,null,900,"""Central and Southern Asia""",1831,"""male""",1990,16862386,1831,16.862386
5,"""World""",null,null,900,"""Eastern and South-Eastern Asia""",1832,"""male""",1990,7069837,1832,7.069837
…,…,…,…,…,…,…,…,…,…,…,…
28026,"""Wallis and Futuna Islands""","""44""","""B R""",876,"""New Caledonia""",540,"""female""",2024,510,540,0.00051
28027,"""Wallis and Futuna Islands""","""44""","""B R""",876,"""Vanuatu""",548,"""female""",2024,28,548,0.000028
28028,"""Wallis and Futuna Islands""","""44""","""B R""",876,"""Polynesia""",957,"""female""",2024,5,957,0.000005


In [6]:
# Now I want to include the region and subregion to the main data set 
migrants = migrants.join(country_codes, on="M49", how="inner")
migrants_from_LAC = migrants.filter(pl.col("sub-region")== "Latin America and the Caribbean")
migrants_from_LAC

index,destination,coverage,type,destination_code,origin,origin_code,gender,year,migrants,M49,migrants_millions,name,alpha-2,alpha-3,country-code,iso_3166-2,region,sub-region,intermediate-region,region-code,sub-region-code,intermediate-region-code
i64,str,str,str,i64,str,i64,str,i32,i64,i64,f64,str,str,str,i64,str,str,str,str,i64,i64,i64
202,"""World""",null,null,900,"""Anguilla""",660,"""male""",1990,1166,660,0.001166,"""Anguilla""","""AI""","""AIA""",660,"""ISO 3166-2:AI""","""Americas""","""Latin America and the Caribbea…","""Caribbean""",19,419,29
203,"""World""",null,null,900,"""Antigua and Barbuda""",28,"""male""",1990,3297,28,0.003297,"""Antigua and Barbuda""","""AG""","""ATG""",28,"""ISO 3166-2:AG""","""Americas""","""Latin America and the Caribbea…","""Caribbean""",19,419,29
204,"""World""",null,null,900,"""Aruba""",533,"""male""",1990,1979,533,0.001979,"""Aruba""","""AW""","""ABW""",533,"""ISO 3166-2:AW""","""Americas""","""Latin America and the Caribbea…","""Caribbean""",19,419,29
205,"""World""",null,null,900,"""Bahamas""",44,"""male""",1990,1017,44,0.001017,"""Bahamas""","""BS""","""BHS""",44,"""ISO 3166-2:BS""","""Americas""","""Latin America and the Caribbea…","""Caribbean""",19,419,29
206,"""World""",null,null,900,"""Barbados""",52,"""male""",1990,9627,52,0.009627,"""Barbados""","""BB""","""BRB""",52,"""ISO 3166-2:BB""","""Americas""","""Latin America and the Caribbea…","""Caribbean""",19,419,29
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
26999,"""New Zealand""","""37""","""B R""",554,"""Chile""",152,"""female""",2024,3025,152,0.003025,"""Chile""","""CL""","""CHL""",152,"""ISO 3166-2:CL""","""Americas""","""Latin America and the Caribbea…","""South America""",19,419,5
27000,"""New Zealand""","""37""","""B R""",554,"""Colombia""",170,"""female""",2024,2440,170,0.00244,"""Colombia""","""CO""","""COL""",170,"""ISO 3166-2:CO""","""Americas""","""Latin America and the Caribbea…","""South America""",19,419,5
27368,"""Micronesia""",null,null,954,"""Puerto Rico""",630,"""female""",2024,2664,630,0.002664,"""Puerto Rico""","""PR""","""PRI""",630,"""ISO 3166-2:PR""","""Americas""","""Latin America and the Caribbea…","""Caribbean""",19,419,29


In [7]:
migrants_from_LAC.write_csv("../Data/migrants_from_LAC.csv")

In [8]:
# Now lets have total migration (not by gender or by destination country)
# We want to have other indicators by country so we need to have the left part of the data base complete

grouped_LAC = (
    migrants_from_LAC
    .filter(pl.col("destination") == "World")
    
    .group_by(["origin", "M49", "alpha-2", "alpha-3",
               "region", "sub-region", "intermediate-region", "year"])
    
    .agg([
        (pl.col("migrants")/1_000_000).sum().alias("total_migrants"),

        # Sum by gender
        pl.when(pl.col("gender") == "female")
          .then(pl.col("migrants")/1_000_000)
          .otherwise(0)
          .sum()
          .alias("female_migrants"),

        pl.when(pl.col("gender") == "male")
          .then(pl.col("migrants")/1_000_000)
          .otherwise(0)
          .sum()
          .alias("male_migrants"),
    ])
    .with_columns([
        (pl.col("female_migrants") / pl.col("total_migrants") * 100)
        .alias("pct_female_migrants"),

        (pl.col("male_migrants") / pl.col("total_migrants") * 100)
        .alias("pct_male_migrants"),
    ])
    .sort(["origin", "year"])
)

grouped_LAC.filter(pl.col("origin") == "Mexico")


origin,M49,alpha-2,alpha-3,region,sub-region,intermediate-region,year,total_migrants,female_migrants,male_migrants,pct_female_migrants,pct_male_migrants
str,i64,str,str,str,str,str,i32,f64,f64,f64,f64,f64
"""Mexico""",484,"""MX""","""MEX""","""Americas""","""Latin America and the Caribbea…","""Central America""",1990,4.588874,1.973414,2.61546,43.004319,56.995681
"""Mexico""",484,"""MX""","""MEX""","""Americas""","""Latin America and the Caribbea…","""Central America""",1995,7.165639,3.231673,3.933966,45.099579,54.900421
"""Mexico""",484,"""MX""","""MEX""","""Americas""","""Latin America and the Caribbea…","""Central America""",2000,9.367663,4.19616,5.171503,44.794096,55.205904
"""Mexico""",484,"""MX""","""MEX""","""Americas""","""Latin America and the Caribbea…","""Central America""",2005,11.329536,5.043952,6.285584,44.520376,55.479624
"""Mexico""",484,"""MX""","""MEX""","""Americas""","""Latin America and the Caribbea…","""Central America""",2010,12.108928,5.647673,6.461255,46.64057,53.35943
"""Mexico""",484,"""MX""","""MEX""","""Americas""","""Latin America and the Caribbea…","""Central America""",2015,12.184226,5.842657,6.341569,47.952632,52.047368
"""Mexico""",484,"""MX""","""MEX""","""Americas""","""Latin America and the Caribbea…","""Central America""",2020,11.452717,5.562762,5.889955,48.571549,51.428451
"""Mexico""",484,"""MX""","""MEX""","""Americas""","""Latin America and the Caribbea…","""Central America""",2024,11.596529,5.665313,5.931216,48.853523,51.146477


In [9]:
import polars as pl

def clean_data_wb(df: pl.DataFrame, variable: str) -> pl.DataFrame:
    """
    Clean a World Bank-style Polars DataFrame by standardizing column names,
    selecting relevant columns, and fixing types.
    """
    df = (
        df.rename({
            "TIME_PERIOD": "year",
            "REF_AREA": "alpha-3",
            "OBS_VALUE": variable
        })
        .select(["year", "alpha-3", variable])
    )

    return df


In [10]:
def world_bank_data(base_df: pl.DataFrame, file_path: str, variable: str):
    new_variable = clean_data_wb(pl.read_csv(Path(file_path), ignore_errors=True), variable)

    complete_df = (
        base_df.join(new_variable, 
        on=["alpha-3", "year"], 
        how="left" )
        .sort(["origin", "year"])
    )

    complete_df.write_csv("../Data/grouped_by_country.csv")

    return complete_df

In [11]:
grouped_LAC = world_bank_data(grouped_LAC, "../Data/WB_Data/WB_KNOMAD_MRI.csv", "remittance_inflows")
grouped_LAC = world_bank_data(grouped_LAC, "../Data/WB_Data/WB_WDI_NY_GDP_PCAP_CD.csv","gdp_per_capita")
grouped_LAC = world_bank_data(grouped_LAC, "../Data/WB_Data/WB_WDI_SP_POP_TOTL.csv", "total_population")
grouped_LAC = world_bank_data(grouped_LAC, "../Data/WB_Data/WB_WDI_SI_POV_GINI.csv", "Gini_index")
grouped_LAC = world_bank_data(grouped_LAC, "../Data/WB_Data/WEF_TTDI_HOMICIDERT.csv", "homicides_per_100k_population")
grouped_LAC = world_bank_data(grouped_LAC, "../Data/WB_Data/WB_CLEAR_EXP_DRT.csv", "exposure_to_droughts")
grouped_LAC = world_bank_data(grouped_LAC, "../Data/WB_Data/Climate_risk_INdex.csv", "climate_risk_index")

grouped_LAC = grouped_LAC.with_columns([
    (pl.col("total_population") / 1_000_000).alias("total_population")
])

grouped_LAC = grouped_LAC.with_columns([
    ((pl.col("total_migrants") / pl.col("total_population"))*100).alias("migration_rate")
])

grouped_LAC.write_csv("../Data/grouped_by_country.csv")

grouped_LAC.filter(pl.col("origin") == "Mexico")

origin,M49,alpha-2,alpha-3,region,sub-region,intermediate-region,year,total_migrants,female_migrants,male_migrants,pct_female_migrants,pct_male_migrants,remittance_inflows,gdp_per_capita,total_population,Gini_index,homicides_per_100k_population,exposure_to_droughts,climate_risk_index,migration_rate
str,i64,str,str,str,str,str,i32,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""Mexico""",484,"""MX""","""MEX""","""Americas""","""Latin America and the Caribbea…","""Central America""",1990,4.588874,1.973414,2.61546,43.004319,56.995681,null,3154.469203,82.82017,null,null,null,null,5.540769
"""Mexico""",484,"""MX""","""MEX""","""Americas""","""Latin America and the Caribbea…","""Central America""",1995,7.165639,3.231673,3.933966,45.099579,54.900421,null,4183.878477,90.862455,null,null,null,null,7.886249
"""Mexico""",484,"""MX""","""MEX""","""Americas""","""Latin America and the Caribbea…","""Central America""",2000,9.367663,4.19616,5.171503,44.794096,55.205904,7524.74298,7524.027138,98.625552,53.4,null,null,null,9.498211
"""Mexico""",484,"""MX""","""MEX""","""Americas""","""Latin America and the Caribbea…","""Central America""",2005,11.329536,5.043952,6.285584,44.520376,55.479624,22741.840918,8671.758919,105.811504,50.9,null,null,null,10.707282
"""Mexico""",484,"""MX""","""MEX""","""Americas""","""Latin America and the Caribbea…","""Central America""",2010,12.108928,5.647673,6.461255,46.64057,53.35943,22767.716708,9728.800782,113.623895,47.7,null,null,null,10.657026
"""Mexico""",484,"""MX""","""MEX""","""Americas""","""Latin America and the Caribbea…","""Central America""",2015,12.184226,5.842657,6.341569,47.952632,52.047368,26824.907167,10021.238612,121.072306,null,null,7.8,null,10.063595
"""Mexico""",484,"""MX""","""MEX""","""Americas""","""Latin America and the Caribbea…","""Central America""",2020,11.452717,5.562762,5.889955,48.571549,51.428451,43977.653965,8841.270751,126.799054,44.6,null,7.2,null,9.032179
"""Mexico""",484,"""MX""","""MEX""","""Americas""","""Latin America and the Caribbea…","""Central America""",2024,11.596529,5.665313,5.931216,48.853523,51.146477,null,14157.944584,130.861007,null,28.175653,null,112.0,8.861715


In [12]:
## I want to show income disparietes as a motivarion to migrate so lets create that data set
## we have 
migrants_from_LAC

index,destination,coverage,type,destination_code,origin,origin_code,gender,year,migrants,M49,migrants_millions,name,alpha-2,alpha-3,country-code,iso_3166-2,region,sub-region,intermediate-region,region-code,sub-region-code,intermediate-region-code
i64,str,str,str,i64,str,i64,str,i32,i64,i64,f64,str,str,str,i64,str,str,str,str,i64,i64,i64
202,"""World""",null,null,900,"""Anguilla""",660,"""male""",1990,1166,660,0.001166,"""Anguilla""","""AI""","""AIA""",660,"""ISO 3166-2:AI""","""Americas""","""Latin America and the Caribbea…","""Caribbean""",19,419,29
203,"""World""",null,null,900,"""Antigua and Barbuda""",28,"""male""",1990,3297,28,0.003297,"""Antigua and Barbuda""","""AG""","""ATG""",28,"""ISO 3166-2:AG""","""Americas""","""Latin America and the Caribbea…","""Caribbean""",19,419,29
204,"""World""",null,null,900,"""Aruba""",533,"""male""",1990,1979,533,0.001979,"""Aruba""","""AW""","""ABW""",533,"""ISO 3166-2:AW""","""Americas""","""Latin America and the Caribbea…","""Caribbean""",19,419,29
205,"""World""",null,null,900,"""Bahamas""",44,"""male""",1990,1017,44,0.001017,"""Bahamas""","""BS""","""BHS""",44,"""ISO 3166-2:BS""","""Americas""","""Latin America and the Caribbea…","""Caribbean""",19,419,29
206,"""World""",null,null,900,"""Barbados""",52,"""male""",1990,9627,52,0.009627,"""Barbados""","""BB""","""BRB""",52,"""ISO 3166-2:BB""","""Americas""","""Latin America and the Caribbea…","""Caribbean""",19,419,29
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
26999,"""New Zealand""","""37""","""B R""",554,"""Chile""",152,"""female""",2024,3025,152,0.003025,"""Chile""","""CL""","""CHL""",152,"""ISO 3166-2:CL""","""Americas""","""Latin America and the Caribbea…","""South America""",19,419,5
27000,"""New Zealand""","""37""","""B R""",554,"""Colombia""",170,"""female""",2024,2440,170,0.00244,"""Colombia""","""CO""","""COL""",170,"""ISO 3166-2:CO""","""Americas""","""Latin America and the Caribbea…","""South America""",19,419,5
27368,"""Micronesia""",null,null,954,"""Puerto Rico""",630,"""female""",2024,2664,630,0.002664,"""Puerto Rico""","""PR""","""PRI""",630,"""ISO 3166-2:PR""","""Americas""","""Latin America and the Caribbea…","""Caribbean""",19,419,29


In [13]:
country_codes_just_alpha_3 = country_codes.select(["M49", "alpha-3"])


In [14]:
# in order to have the GDP of destination countries we need to add the code of destination 

migrants_from_LAC = (
    migrants_from_LAC.join(
    country_codes_just_alpha_3.with_columns(pl.col("M49").cast(pl.Int64)),
    left_on="destination_code",
    right_on="M49",
    how="left",
    suffix="_dest"
))

In [15]:
migrants_from_LAC

index,destination,coverage,type,destination_code,origin,origin_code,gender,year,migrants,M49,migrants_millions,name,alpha-2,alpha-3,country-code,iso_3166-2,region,sub-region,intermediate-region,region-code,sub-region-code,intermediate-region-code,alpha-3_dest
i64,str,str,str,i64,str,i64,str,i32,i64,i64,f64,str,str,str,i64,str,str,str,str,i64,i64,i64,str
202,"""World""",null,null,900,"""Anguilla""",660,"""male""",1990,1166,660,0.001166,"""Anguilla""","""AI""","""AIA""",660,"""ISO 3166-2:AI""","""Americas""","""Latin America and the Caribbea…","""Caribbean""",19,419,29,null
203,"""World""",null,null,900,"""Antigua and Barbuda""",28,"""male""",1990,3297,28,0.003297,"""Antigua and Barbuda""","""AG""","""ATG""",28,"""ISO 3166-2:AG""","""Americas""","""Latin America and the Caribbea…","""Caribbean""",19,419,29,null
204,"""World""",null,null,900,"""Aruba""",533,"""male""",1990,1979,533,0.001979,"""Aruba""","""AW""","""ABW""",533,"""ISO 3166-2:AW""","""Americas""","""Latin America and the Caribbea…","""Caribbean""",19,419,29,null
205,"""World""",null,null,900,"""Bahamas""",44,"""male""",1990,1017,44,0.001017,"""Bahamas""","""BS""","""BHS""",44,"""ISO 3166-2:BS""","""Americas""","""Latin America and the Caribbea…","""Caribbean""",19,419,29,null
206,"""World""",null,null,900,"""Barbados""",52,"""male""",1990,9627,52,0.009627,"""Barbados""","""BB""","""BRB""",52,"""ISO 3166-2:BB""","""Americas""","""Latin America and the Caribbea…","""Caribbean""",19,419,29,null
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
26999,"""New Zealand""","""37""","""B R""",554,"""Chile""",152,"""female""",2024,3025,152,0.003025,"""Chile""","""CL""","""CHL""",152,"""ISO 3166-2:CL""","""Americas""","""Latin America and the Caribbea…","""South America""",19,419,5,"""NZL"""
27000,"""New Zealand""","""37""","""B R""",554,"""Colombia""",170,"""female""",2024,2440,170,0.00244,"""Colombia""","""CO""","""COL""",170,"""ISO 3166-2:CO""","""Americas""","""Latin America and the Caribbea…","""South America""",19,419,5,"""NZL"""
27368,"""Micronesia""",null,null,954,"""Puerto Rico""",630,"""female""",2024,2664,630,0.002664,"""Puerto Rico""","""PR""","""PRI""",630,"""ISO 3166-2:PR""","""Americas""","""Latin America and the Caribbea…","""Caribbean""",19,419,29,null


In [16]:
def world_bank_data_dest(base_df: pl.DataFrame, file_path: str, variable: str):
    new_variable = pl.read_csv(Path(file_path), ignore_errors=True)
    
    new_variable = (
        new_variable.rename({
            "TIME_PERIOD": "year",
            "REF_AREA": "alpha-3_dest",
            "OBS_VALUE": variable
        })
        .select(["year", "alpha-3_dest", variable])
    )
    
    complete_df = (
        base_df.join(
            new_variable,
            on=["alpha-3_dest", "year"],
            how="left",
            suffix="_dest"
        )
        .sort(["origin", "year"])
    )

    complete_df.write_csv("../Data/migrants_from_LAC.csv")

    return complete_df

In [17]:
migrants_from_LAC = migrants_from_LAC.filter(pl.col("alpha-3_dest").is_not_null())
migrants_from_LAC

index,destination,coverage,type,destination_code,origin,origin_code,gender,year,migrants,M49,migrants_millions,name,alpha-2,alpha-3,country-code,iso_3166-2,region,sub-region,intermediate-region,region-code,sub-region-code,intermediate-region-code,alpha-3_dest
i64,str,str,str,i64,str,i64,str,i32,i64,i64,f64,str,str,str,i64,str,str,str,str,i64,i64,i64,str
6938,"""Seychelles""",null,"""B""",690,"""Cuba""",192,"""male""",1990,9,192,0.000009,"""Cuba""","""CU""","""CUB""",192,"""ISO 3166-2:CU""","""Americas""","""Latin America and the Caribbea…","""Caribbean""",19,419,29,"""SYC"""
7536,"""Congo""",null,"""B R""",178,"""Cuba""",192,"""male""",1990,49,192,0.000049,"""Cuba""","""CU""","""CUB""",192,"""ISO 3166-2:CU""","""Americas""","""Latin America and the Caribbea…","""Caribbean""",19,419,29,"""COG"""
7997,"""Egypt""",null,"""B R""",818,"""Brazil""",76,"""male""",1990,25,76,0.000025,"""Brazil""","""BR""","""BRA""",76,"""ISO 3166-2:BR""","""Americas""","""Latin America and the Caribbea…","""South America""",19,419,5,"""EGY"""
8128,"""Libya""",null,"""C R""",434,"""El Salvador""",222,"""male""",1990,188,222,0.000188,"""El Salvador""","""SV""","""SLV""",222,"""ISO 3166-2:SV""","""Americas""","""Latin America and the Caribbea…","""Central America""",19,419,13,"""LBY"""
8130,"""Libya""",null,"""C R""",434,"""Brazil""",76,"""male""",1990,157,76,0.000157,"""Brazil""","""BR""","""BRA""",76,"""ISO 3166-2:BR""","""Americas""","""Latin America and the Caribbea…","""South America""",19,419,5,"""LBY"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
26998,"""New Zealand""","""37""","""B R""",554,"""Brazil""",76,"""female""",2024,5788,76,0.005788,"""Brazil""","""BR""","""BRA""",76,"""ISO 3166-2:BR""","""Americas""","""Latin America and the Caribbea…","""South America""",19,419,5,"""NZL"""
26999,"""New Zealand""","""37""","""B R""",554,"""Chile""",152,"""female""",2024,3025,152,0.003025,"""Chile""","""CL""","""CHL""",152,"""ISO 3166-2:CL""","""Americas""","""Latin America and the Caribbea…","""South America""",19,419,5,"""NZL"""
27000,"""New Zealand""","""37""","""B R""",554,"""Colombia""",170,"""female""",2024,2440,170,0.00244,"""Colombia""","""CO""","""COL""",170,"""ISO 3166-2:CO""","""Americas""","""Latin America and the Caribbea…","""South America""",19,419,5,"""NZL"""


In [18]:
migrants_from_LAC = world_bank_data_dest(migrants_from_LAC, "../Data/WB_Data/WB_WDI_NY_GDP_PCAP_CD.csv","gdp_per_capita_dest")
migrants_from_LAC.filter(pl.col("origin") == "Mexico", pl.col("year") == 2024)

index,destination,coverage,type,destination_code,origin,origin_code,gender,year,migrants,M49,migrants_millions,name,alpha-2,alpha-3,country-code,iso_3166-2,region,sub-region,intermediate-region,region-code,sub-region-code,intermediate-region-code,alpha-3_dest,gdp_per_capita_dest
i64,str,str,str,i64,str,i64,str,i32,i64,i64,f64,str,str,str,i64,str,str,str,str,i64,i64,i64,str,f64
19242,"""Portugal""",null,"""B""",620,"""Mexico""",484,"""female""",2024,189,484,0.000189,"""Mexico""","""MX""","""MEX""",484,"""ISO 3166-2:MX""","""Americas""","""Latin America and the Caribbea…","""Central America""",19,419,13,"""PRT""",28844.497925
20440,"""Liechtenstein""",null,"""B""",438,"""Mexico""",484,"""female""",2024,23,484,0.000023,"""Mexico""","""MX""","""MEX""",484,"""ISO 3166-2:MX""","""Americas""","""Latin America and the Caribbea…","""Central America""",19,419,13,"""LIE""",null
21602,"""Bahamas""",null,"""B R""",44,"""Mexico""",484,"""male""",2024,97,484,0.000097,"""Mexico""","""MX""","""MEX""",484,"""ISO 3166-2:MX""","""Americas""","""Latin America and the Caribbea…","""Central America""",19,419,13,"""BHS""",39455.446655
25394,"""Uruguay""",null,"""B R""",858,"""Mexico""",484,"""female""",2024,445,484,0.000445,"""Mexico""","""MX""","""MEX""",484,"""ISO 3166-2:MX""","""Americas""","""Latin America and the Caribbea…","""Central America""",19,419,13,"""URY""",23906.513303
22179,"""Dominican Republic""",null,"""B R""",214,"""Mexico""",484,"""female""",2024,807,484,0.000807,"""Mexico""","""MX""","""MEX""",484,"""ISO 3166-2:MX""","""Americas""","""Latin America and the Caribbea…","""Central America""",19,419,13,"""DOM""",10875.661844
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
26878,"""Australia""",null,"""B R""",36,"""Mexico""",484,"""female""",2024,4937,484,0.004937,"""Mexico""","""MX""","""MEX""",484,"""ISO 3166-2:MX""","""Americas""","""Latin America and the Caribbea…","""Central America""",19,419,13,"""AUS""",64407.484257
14696,"""Bulgaria""",null,"""B""",100,"""Mexico""",484,"""male""",2024,67,484,0.000067,"""Mexico""","""MX""","""MEX""",484,"""ISO 3166-2:MX""","""Americas""","""Latin America and the Caribbea…","""Central America""",19,419,13,"""BGR""",17412.411509
20663,"""Luxembourg""",null,"""B""",442,"""Mexico""",484,"""female""",2024,283,484,0.000283,"""Mexico""","""MX""","""MEX""",484,"""ISO 3166-2:MX""","""Americas""","""Latin America and the Caribbea…","""Central America""",19,419,13,"""LUX""",137516.587324


In [19]:
migrants_from_LAC = world_bank_data_dest(migrants_from_LAC, "../Data/WB_Data/WB_WDI_SP_POP_TOTL.csv", "total_population_dest")

migrants_from_LAC  = migrants_from_LAC .with_columns([
    (pl.col("total_population_dest") / 1_000_000).alias("total_population_dest")
])

migrants_from_LAC.sort(["origin", "year"]).filter(pl.col("origin") == "Mexico", pl.col("year") == 2024)

index,destination,coverage,type,destination_code,origin,origin_code,gender,year,migrants,M49,migrants_millions,name,alpha-2,alpha-3,country-code,iso_3166-2,region,sub-region,intermediate-region,region-code,sub-region-code,intermediate-region-code,alpha-3_dest,gdp_per_capita_dest,total_population_dest
i64,str,str,str,i64,str,i64,str,i32,i64,i64,f64,str,str,str,i64,str,str,str,str,i64,i64,i64,str,f64,f64
19242,"""Portugal""",null,"""B""",620,"""Mexico""",484,"""female""",2024,189,484,0.000189,"""Mexico""","""MX""","""MEX""",484,"""ISO 3166-2:MX""","""Americas""","""Latin America and the Caribbea…","""Central America""",19,419,13,"""PRT""",28844.497925,10.701636
20440,"""Liechtenstein""",null,"""B""",438,"""Mexico""",484,"""female""",2024,23,484,0.000023,"""Mexico""","""MX""","""MEX""",484,"""ISO 3166-2:MX""","""Americas""","""Latin America and the Caribbea…","""Central America""",19,419,13,"""LIE""",null,0.040197
21602,"""Bahamas""",null,"""B R""",44,"""Mexico""",484,"""male""",2024,97,484,0.000097,"""Mexico""","""MX""","""MEX""",484,"""ISO 3166-2:MX""","""Americas""","""Latin America and the Caribbea…","""Central America""",19,419,13,"""BHS""",39455.446655,0.401283
25394,"""Uruguay""",null,"""B R""",858,"""Mexico""",484,"""female""",2024,445,484,0.000445,"""Mexico""","""MX""","""MEX""",484,"""ISO 3166-2:MX""","""Americas""","""Latin America and the Caribbea…","""Central America""",19,419,13,"""URY""",23906.513303,3.386588
22179,"""Dominican Republic""",null,"""B R""",214,"""Mexico""",484,"""female""",2024,807,484,0.000807,"""Mexico""","""MX""","""MEX""",484,"""ISO 3166-2:MX""","""Americas""","""Latin America and the Caribbea…","""Central America""",19,419,13,"""DOM""",10875.661844,11.427557
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
26878,"""Australia""",null,"""B R""",36,"""Mexico""",484,"""female""",2024,4937,484,0.004937,"""Mexico""","""MX""","""MEX""",484,"""ISO 3166-2:MX""","""Americas""","""Latin America and the Caribbea…","""Central America""",19,419,13,"""AUS""",64407.484257,27.204809
14696,"""Bulgaria""",null,"""B""",100,"""Mexico""",484,"""male""",2024,67,484,0.000067,"""Mexico""","""MX""","""MEX""",484,"""ISO 3166-2:MX""","""Americas""","""Latin America and the Caribbea…","""Central America""",19,419,13,"""BGR""",17412.411509,6.444366
20663,"""Luxembourg""",null,"""B""",442,"""Mexico""",484,"""female""",2024,283,484,0.000283,"""Mexico""","""MX""","""MEX""",484,"""ISO 3166-2:MX""","""Americas""","""Latin America and the Caribbea…","""Central America""",19,419,13,"""LUX""",137516.587324,0.677717


In [20]:
import polars as pl

df_2024 = migrants_from_LAC.filter(pl.col("year") == 2024)

# Compute the weighted value (GDP × population)
df_2024 = df_2024.with_columns(
    (pl.col("gdp_per_capita_dest") * pl.col("total_population_dest")).alias("weighted_gdp_dest")
)

# Group by origin and sum
df_2024 = (
    df_2024
    .group_by("origin","alpha-3","year" )
    .agg([
        pl.col("total_population_dest").sum().alias("total_pop_dest_sum"),
        pl.col("weighted_gdp_dest").sum().alias("weighted_gdp_dest_sum")
    ])
)

# Compute weighted average of destination GDP
df_2024 = df_2024.with_columns(
    (pl.col("weighted_gdp_dest_sum") / pl.col("total_pop_dest_sum")).alias("weighted_avg_dest_gdp")
)

df_2024.filter(pl.col("origin") == "Mexico")

origin,alpha-3,year,total_pop_dest_sum,weighted_gdp_dest_sum,weighted_avg_dest_gdp
str,str,i32,f64,f64,f64
"""Mexico""","""MEX""",2024,5542.51723,1.4035e8,25322.255083


In [21]:
grouped_LAC = grouped_LAC.join(
    df_2024.select(["alpha-3", "year", "weighted_avg_dest_gdp"]),
    on=["alpha-3", "year"],
    how="left"
)
grouped_LAC

grouped_LAC.write_csv("../Data/grouped_by_country.csv")

grouped_LAC.filter(pl.col("origin") == "Mexico", pl.col("year") == 2024)

origin,M49,alpha-2,alpha-3,region,sub-region,intermediate-region,year,total_migrants,female_migrants,male_migrants,pct_female_migrants,pct_male_migrants,remittance_inflows,gdp_per_capita,total_population,Gini_index,homicides_per_100k_population,exposure_to_droughts,climate_risk_index,migration_rate,weighted_avg_dest_gdp
str,i64,str,str,str,str,str,i32,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""Mexico""",484,"""MX""","""MEX""","""Americas""","""Latin America and the Caribbea…","""Central America""",2024,11.596529,5.665313,5.931216,48.853523,51.146477,null,14157.944584,130.861007,null,28.175653,null,112.0,8.861715,25322.255083
